# 1. Data pipeline

Tokenisation, dependency parsing, constituency parsing (new for HAABSA7GCN),
and the `ABSADataset` that batches everything for training.

  - `Tokenizer4Bert` (cell 3) — wraps `BertTokenizer` from `pytorch_transformers`.
  - `DepInstanceParser` (cell 4) — reads `.dep` files into the typed dep adjacency.
  - `ABSADataset` (cell 6) — per-instance feature builder. Returns tensors for
    every GCN module: `dep_adj_matrix` (SynGCN), `dep_adj_matrix_knogcn`
    (KnoGCN), and (new) `const_adj_matrix` (ConstGCN). LexGCN and SemGCN
    build their adjacencies inside `AsaTgcnSem.forward`, not here.

Constituency trees are produced offline by `src/build_const_trees.py` and
stored as `data/{train,test}{year}restaurant.txt.const` next to the existing
`.dep` files; `ABSADataset.load_constfile` reads them at dataset-build time
when `opt.modules['constgcn']` is true.

In [2]:
import os
import copy
import numpy as np
import torch
from torch.utils.data import Dataset
# !pip install pytorch_transformers
from pytorch_transformers import BertTokenizer
import gc

C:\Users\bromi\anaconda3\envs\trigcn\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def pad_and_truncate(sequence, maxlen, dtype='int64', padding='post', truncating='post', value=0):
#is designed to adjust the length of a sequence (such as a list or array) to a specified maximum length
#by either padding or truncating the sequence as necessary
#padding='post' add words in the end of the sentence if necessary 
#trancating='post' trancate sentence in the end if the length of a sentence is longer than maxlen
    
    x = (np.ones(maxlen) * value).astype(dtype)
    
    if truncating == 'pre':
        trunc = sequence[-maxlen:] 
    else:
        trunc = sequence[:maxlen]
    
    trunc = np.asarray(trunc, dtype=dtype)

    if padding == 'post':
        x[:len(trunc)] = trunc
    else:
        x[-len(trunc):] = trunc
    return x

In [4]:
class Tokenizer4Bert:
#is designed to handle the tokenization of text for use with a BERT model
    def __init__(self, max_seq_len, pretrained_bert_name):
        self.tokenizer = BertTokenizer.from_pretrained(pretrained_bert_name)
        self.max_seq_len = max_seq_len

    def text_to_sequence(self, text, reverse=False, padding='post', truncating='post'):
        sequence = self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(text)) 
        #tokenize the text and thenmap it with the corresponding ids
        if len(sequence) == 0:
            sequence = [0]
        if reverse:
            sequence = sequence[::-1]
        return pad_and_truncate(sequence, self.max_seq_len, padding=padding, truncating=truncating)

    def id_to_sequence(self, sequence, reverse=False, padding='post', truncating='post'):
        if len(sequence) == 0:
            sequence = [0]
        if reverse:
            sequence = sequence[::-1]
        return pad_and_truncate(sequence, self.max_seq_len, padding=padding, truncating=truncating)

In [5]:
class DepInstanceParser():
    def __init__(self, basicDependencies, tokens):
        self.basicDependencies = basicDependencies
        self.tokens = tokens
        self.words = []
        self.dep_governed_info = []
        self.dep_parsing()


    def dep_parsing(self):
        if len(self.tokens) > 0:
            words = []
            for token in self.tokens:
                token['word'] = token
                words.append(self.change_word(token['word'])) 
            dep_governed_info = [
                {"word": word}
                for i,word in enumerate(words)
            ]
            self.words = words
        else:
            dep_governed_info = [{}] * len(self.basicDependencies)
        for dep in self.basicDependencies:
            dependent_index = dep['dependent'] - 1
            governed_index = dep['governor'] - 1
            dep_governed_info[dependent_index] = {
                "governor": governed_index,
                "dep": dep['dep']
            }
        self.dep_governed_info = dep_governed_info #contains detailed information about the dependencies among these tokens.
    def change_word(self, word):
    #designed to handle specific formatting issues within the text data it processes, particularly dealing 
    #with tokens representing left and right parentheses.
        
        if "-RRB-" in word:
        #The method first checks if the string "-RRB-" is present in the word. This string is often used in 
        #linguistic data to represent a right parenthesis ) to prevent misinterpretation during parsing processes. 
        #If "-RRB-" is found, it is replaced with ")".
            return word.replace("-RRB-", ")")
        
        if "-LRB-" in word:
        #Next, the method checks for the presence of "-LRB-" in the word. Similarly, this string represents a left 
        #parenthesis ( and is replaced by "(".
            return word.replace("-LRB-", "(")
        return word

    def get_first_order(self, direct=False):
        #designed to generate matrices representing the adjacency and types of dependency relationships between 
        #tokens in a sentence based on their parsed dependencies.
        
        #indicate whether there is a direct dependency link between the tokens
        dep_adj_matrix  = [[0] * len(self.dep_governed_info) for _ in range(len(self.dep_governed_info))]
        
        #indicate the type of dependency (like "subj", "obj") between tokens instead of binary indicators as in dep_adj_matrix
        dep_type_matrix = [["none"] * len(self.dep_governed_info) for _ in range(len(self.dep_governed_info))]
        
        
        for i, dep_info in enumerate(self.dep_governed_info):
            governor = dep_info["governor"] #the index of the token that governs the current token
            dep_type = dep_info["dep"] #the type of the dependency
            
            #indicate the existance of the dependency between tokens
            dep_adj_matrix[i][governor] = 1
            dep_adj_matrix[governor][i] = 1
            
            #If direct is False, both [i][governor] and [governor][i] are set to the dependency type.
            #If direct is True, the entries are suffixed to indicate the direction (_in for incoming, _out for outgoing 
            #dependencies relative to each token).
            
            dep_type_matrix[i][governor] = dep_type if direct is False else "{}_in".format(dep_type)
            dep_type_matrix[governor][i] = dep_type if direct is False else "{}_out".format(dep_type)
        
        return dep_adj_matrix, dep_type_matrix

    def get_next_order(self, dep_adj_matrix, dep_type_matrix):
        new_dep_adj_matrix = copy.deepcopy(dep_adj_matrix)
        new_dep_type_matrix = copy.deepcopy(dep_type_matrix)
        for target_index in range(len(dep_adj_matrix)):
            for first_order_index in range(len(dep_adj_matrix[target_index])):
                if dep_adj_matrix[target_index][first_order_index] == 0:
                    continue
                for second_order_index in range(len(dep_adj_matrix[first_order_index])):
                    if dep_adj_matrix[first_order_index][second_order_index] == 0:
                        continue
                    if second_order_index == target_index:
                        continue
                    if new_dep_adj_matrix[target_index][second_order_index] == 1:
                        continue
                    new_dep_adj_matrix[target_index][second_order_index] = 1
                    new_dep_type_matrix[target_index][second_order_index] = dep_type_matrix[first_order_index][second_order_index]
        return new_dep_adj_matrix, new_dep_type_matrix

    def get_second_order(self, direct=False):
        dep_adj_matrix, dep_type_matrix = self.get_first_order(direct=direct)
        return self.get_next_order(dep_adj_matrix, dep_type_matrix)

    def get_third_order(self, direct=False):
        dep_adj_matrix, dep_type_matrix = self.get_second_order(direct=direct)
        return self.get_next_order(dep_adj_matrix, dep_type_matrix)

    def search_dep_path(self, start_idx, end_idx, adj_max, dep_path_arr):
        for next_id in range(len(adj_max[start_idx])):
            if next_id in dep_path_arr or adj_max[start_idx][next_id] in ["none"]:
                continue
            if next_id == end_idx:
                return 1, dep_path_arr + [next_id]
            stat, dep_arr = self.search_dep_path(next_id, end_idx, adj_max, dep_path_arr + [next_id])
            if stat == 1:
                return stat, dep_arr
        return 0, []

    def get_dep_path(self, start_index, end_index, direct=False):
        dep_adj_matrix, dep_type_matrix = self.get_first_order(direct=direct)
        _, dep_path = self.search_dep_path(start_index, end_index, dep_type_matrix, [start_index])
        return dep_path

In [6]:
class DefaultConfig:
    def __init__(self):
        self.print_sent = False
        self.max_seq_len = 256 

def get_default_config():
    return DefaultConfig()

In [ ]:
# Constituency parsing helpers for ConstGCN
# CONST_LABEL_VOCAB lists the edge-type IDs used by ConstGCN.
# Indices 2-7 are the Penn Treebank phrase categories in P
CONST_LABEL_VOCAB = ("none", "self", "NP", "VP", "ADJP", "ADVP", "PP", "SBAR")
CONST_PHRASE_SET = frozenset(CONST_LABEL_VOCAB[2:])
constlabel2id = {lab: i for i, lab in enumerate(CONST_LABEL_VOCAB)}


def _parse_ptb_tree(tree_str):
    toks = tree_str.replace("(", " ( ").replace(")", " ) ").split()
    pos = [0]
    def _parse():
        assert toks[pos[0]] == "("
        pos[0] += 1
        label = toks[pos[0]]
        pos[0] += 1
        children = []
        while toks[pos[0]] != ")":
            if toks[pos[0]] == "(":
                children.append(_parse())
            else:
                children.append(toks[pos[0]])
                pos[0] += 1
        pos[0] += 1
        return (label, children)
    return _parse()


def _leaf_internal_paths(tree):
    paths = []
    counter = [0]
    def _walk(node, path):
        if isinstance(node, str):
            paths.append(list(path))
            return
        label, children = node
        node_id = counter[0]
        counter[0] += 1
        new_path = path + [(node_id, label)]
        for child in children:
            _walk(child, new_path)
    _walk(tree, [])
    return paths


def build_const_word_matrices(tree_str, n_expected=None):
    tree = _parse_ptb_tree(tree_str)
    paths = _leaf_internal_paths(tree)
    n = len(paths)
    if n_expected is not None and n != n_expected:
        raise ValueError(
            f"const tree leaf count {n} does not match expected word count "
            f"{n_expected} - tokenisation drift between build_const_trees.py "
            f"and ABSADataset.ws()"
        )
    adj = [[0] * n for _ in range(n)]
    labs = [["none"] * n for _ in range(n)]
    for i in range(n):
        adj[i][i] = 1
        labs[i][i] = "self"
    for i in range(n):
        for j in range(i + 1, n):
            lca_label = None
            for k in range(min(len(paths[i]), len(paths[j]))):
                if paths[i][k][0] != paths[j][k][0]:
                    break
                lca_label = paths[i][k][1]
            if lca_label in CONST_PHRASE_SET:
                adj[i][j] = adj[j][i] = 1
                labs[i][j] = labs[j][i] = lca_label
    return adj, labs


# ABSADataset, extended with ConstGCN tensors
class ABSADataset(Dataset):
    def __init__(self, datafile, tokenizer, opt, deptype2id=None, dep_order="first"):
        self.datafile = datafile
        self.depfile = "{}.dep".format(datafile)
        self.constfile = "{}.const".format(datafile)
        self.tokenizer = tokenizer
        self.opt = opt
        self.deptype2id = deptype2id
        self.dep_order = dep_order
        self.textdata = ABSADataset.load_datafile(self.datafile)
        self.depinfo = ABSADataset.load_depfile(self.depfile)
        self.polarity2id = self.get_polarity2id()
        self.feature = []
        self.use_knogcn = opt.modules['knogcn']
        # optional constituency tree load (skipped when flag is off)
        self.use_constgcn = opt.modules.get('constgcn', False)
        if self.use_constgcn:
            self.consttrees = ABSADataset.load_constfile(self.constfile)
            if len(self.consttrees) != len(self.textdata):
                raise ValueError(
                    f"{self.constfile} has {len(self.consttrees)} parses but "
                    f"{self.datafile} has {len(self.textdata)} instances - "
                    f"rerun build_const_trees.py"
                )
        else:
            self.consttrees = [None] * len(self.textdata)
        # XCatGCN per-instance fields. Frozen pretrained-BERT pool x_m
        # is supplied by opt.cross_graphs (a CrossExampleGraphs instance);
        # split name is auto-derived from the data filename so existing
        # callsites do not change.
        self.use_xcatgcn = opt.modules.get('xcatgcn', False)
        # Either cross-example module needs the cross_graphs object;
        # the dataset binds it once whenever at least one is active.
        self.use_xsimgcn = opt.modules.get('xsimgcn', False)
        if self.use_xcatgcn or self.use_xsimgcn:
            self.cross_graphs = opt.cross_graphs
            fname = os.path.basename(self.datafile).lower()
            if fname.startswith('train'):
                self.split_name = 'train'
            elif fname.startswith('test'):
                self.split_name = 'test'
            else:
                raise ValueError(
                    f"can not auto-derive split from {self.datafile!r}; expected 'train*' or 'test*'"
                )
            split_M = self.cross_graphs.split_X(self.split_name).size(0)
            if split_M != len(self.textdata):
                raise ValueError(
                    f"cross_graphs.{self.split_name} has {split_M} mentions but {self.datafile} "
                    f"has {len(self.textdata)} instances - rerun build_cross_features.py"
                )
        for mention_id, (sentence, depinfo, consttree) in enumerate(zip(self.textdata, self.depinfo, self.consttrees)):
            self.feature.append(self.create_feature(sentence, depinfo, consttree, opt.print_sent, mention_id=mention_id))

        #print(self.feature[:1])

    def __getitem__(self, index):
        return self.feature[index]

    def __len__(self):
        return len(self.feature)

    def ws(self, text):
        tokens = []
        valid_ids = []
        for i, word in enumerate(text):
            if len(text) <= 0:
                continue
            token = self.tokenizer.tokenizer.tokenize(word)
            tokens.extend(token)
            for m in range(len(token)):
                if m == 0:
                    valid_ids.append(1)
                else:
                    valid_ids.append(0)
        token_ids = self.tokenizer.tokenizer.convert_tokens_to_ids(tokens)
        return tokens, token_ids, valid_ids

    def create_feature(self, sentence, depinfo, consttree=None, print_sent = False, mention_id=0):  # consttree, mention_id args

        text_left, text_right, aspect, polarity = sentence

        cls_id = self.tokenizer.tokenizer.vocab["[CLS]"]

        sep_id = self.tokenizer.tokenizer.vocab["[SEP]"]


        doc = text_left + " " + aspect + " " + text_right

        left_tokens, left_token_ids, left_valid_ids = self.ws(text_left.split(" "))

        right_tokens, right_token_ids, right_valid_ids = self.ws(text_right.split(" "))

        aspect_tokens, aspect_token_ids, aspect_valid_ids = self.ws(aspect.split(" "))

        tokens = left_tokens + aspect_tokens + right_tokens

        input_ids = [cls_id] + left_token_ids + aspect_token_ids + right_token_ids + [sep_id] + aspect_token_ids + [sep_id]
        valid_ids = [1] + left_valid_ids + aspect_valid_ids + right_valid_ids + [1] + aspect_valid_ids + [1]
        mem_valid_ids = [0] + [0] * len(left_tokens) + [1] * len(aspect_tokens) + [0] * len(right_tokens) # aspect terms mask

        segment_ids = [0] * (len(tokens) + 2) + [1] * (len(aspect_tokens)+1)


        dep_instance_parser = DepInstanceParser(basicDependencies=depinfo, tokens=[])
        if self.dep_order == "first":
            dep_adj_matrix, dep_type_matrix = dep_instance_parser.get_first_order()
        elif self.dep_order == "second":
            dep_adj_matrix, dep_type_matrix = dep_instance_parser.get_second_order()
        elif self.dep_order == "third":
            dep_adj_matrix, dep_type_matrix = dep_instance_parser.get_third_order()
        else:
            raise ValueError()


        token_head_list = []

        for input_id, valid_id in zip(input_ids, valid_ids):
            if input_id == cls_id:
                continue
            if input_id == sep_id:
                break
            if valid_id == 1:
                token_head_list.append(input_id)

        dep_adj_matrix_knogcn=copy.deepcopy(dep_adj_matrix)
        dep_type_matrix_knogcn=copy.deepcopy(dep_type_matrix)

        if self.use_knogcn:
            self.onto_words=onto_words

            for i in range(len(token_head_list)):
                check=token_head_list[i] in self.onto_words
                if not check:
                    for j in range(len(dep_adj_matrix[i])):
                        dep_adj_matrix_knogcn[i][j]=0
                        dep_type_matrix_knogcn[i][j]='none'


        input_ids = self.tokenizer.id_to_sequence(input_ids)
        valid_ids = self.tokenizer.id_to_sequence(valid_ids)
        segment_ids = self.tokenizer.id_to_sequence(segment_ids)
        mem_valid_ids = self.tokenizer.id_to_sequence(mem_valid_ids)

        size = input_ids.shape[0]

        if print_sent:
            print(doc)
            print(len(dep_adj_matrix[0]))


        # column index offset by 1 to
        # align with [CLS] at BERT subword position 0 (matching the row's i+1 offset). 
        # Corrects sentences where every word maps to a single
        # BERT subword. For multi-subword words the row index also drifts
        # (word i's row at i+1 is wrong if a preceding word was multi-
        # subword); that further fix needs first-subword-position tracking
        # via the valid_ids mask and is retained as a known limitation.
        final_dep_adj_matrix = [[0] * size for _ in range(size)]
        final_dep_value_matrix = [[0] * size for _ in range(size)]

        for i in range(len(token_head_list)):
            if i + 1 >= size:
                break
            for j in range(len(dep_adj_matrix[i])):
                if j + 1 >= size:
                    break
                final_dep_adj_matrix[i+1][j+1] = dep_adj_matrix[i][j]
                final_dep_value_matrix[i+1][j+1] = self.deptype2id[dep_type_matrix[i][j]]

        final_dep_adj_matrix_knogcn = [[0] * size for _ in range(size)]
        final_dep_value_matrix_knogcn = [[0] * size for _ in range(size)]

        for i in range(len(token_head_list)):
            if i + 1 >= size:
                break
            for j in range(len(dep_adj_matrix_knogcn[i])):
                if j + 1 >= size:
                    break
                final_dep_adj_matrix_knogcn[i+1][j+1] = dep_adj_matrix_knogcn[i][j]
                final_dep_value_matrix_knogcn[i+1][j+1] = self.deptype2id[dep_type_matrix_knogcn[i][j]]


        # ConstGCN matrices. Word->subword projection mirrors the SynGCN / KnoGCN projection pattern
        final_const_adj_matrix = [[0] * size for _ in range(size)]
        final_const_value_matrix = [[0] * size for _ in range(size)]
        if self.use_constgcn and consttree is not None:
            const_adj_word, const_lab_word = build_const_word_matrices(
                consttree, n_expected=len(token_head_list)
            )
            for i in range(len(token_head_list)):
                if i + 1 >= size:
                    break
                for j in range(len(const_adj_word[i])):
                    if j + 1 >= size:
                        break
                    final_const_adj_matrix[i+1][j+1] = const_adj_word[i][j]
                    final_const_value_matrix[i+1][j+1] = constlabel2id[const_lab_word[i][j]]


        feature = {
            "input_ids":torch.tensor(input_ids),
            "valid_ids":torch.tensor(valid_ids),
            "segment_ids":torch.tensor(segment_ids),
            "mem_valid_ids":torch.tensor(mem_valid_ids),
            "dep_adj_matrix":torch.tensor(final_dep_adj_matrix),
            "dep_value_matrix":torch.tensor(final_dep_value_matrix),
            "dep_adj_matrix_knogcn":torch.tensor(final_dep_adj_matrix_knogcn),
            "dep_value_matrix_knogcn":torch.tensor(final_dep_value_matrix_knogcn),
            "const_adj_matrix":torch.tensor(final_const_adj_matrix),
            "const_value_matrix":torch.tensor(final_const_value_matrix),
            "polarity": self.polarity2id[polarity],
            "raw_text": doc,
            "aspect": aspect
        }
        # XCatGCN per-instance fields. Stored only when enabled
        # so 4/5GCN runs are unaffected. Self-feature is the frozen
        # pretrained-BERT pool x_m for this mention; category_id and
        # mention_id let the model look up the same-category neighbour
        # pool (with self-exclusion for train) at forward time.
        # Emit the xcat_* fields whenever EITHER cross-example module
        # is active. XSim reuses the same self_feature / mention_id / split_indicator
        # (xcat_category_id is XCat-only but harmless to emit when XSim is on).
        if self.use_xcatgcn or self.use_xsimgcn:
            x_m = self.cross_graphs.split_X(self.split_name)[mention_id]
            feature["xcat_self_feature"] = x_m.detach().clone()
            feature["xcat_category_id"] = torch.tensor(
                self.cross_graphs.category_id(self.split_name, mention_id), dtype=torch.long
            )
            feature["xcat_mention_id"] = torch.tensor(mention_id, dtype=torch.long)
            feature["xcat_split_indicator"] = torch.tensor(
                # 3-state: 0=train, 1=val, 2=test.
                # Dataset builder only ever sees 'train' or 'test' at split_name;
                # val instances get retagged to 1 after random_split (see Instructor).
                {'train': 0, 'test': 2}[self.split_name], dtype=torch.long
            )
        return feature


    @staticmethod
    def load_depfile(filename):
        data = []
        with open(filename, 'r') as f:
            dep_info = []
            for line in f:
                line = line.strip()
                if len(line) > 0:
                    items = line.split("\t")
                    dep_info.append({
                        "governor": int(items[0]),
                        "dependent": int(items[1]),
                        "dep": items[2],
                    })
                else:
                    if len(dep_info) > 0:
                        data.append(dep_info)
                        dep_info = []
            if len(dep_info) > 0:
                data.append(dep_info)
                dep_info = []
        return data

    @staticmethod
    def load_constfile(filename):
        trees = []
        with open(filename, "r") as f:
            buf = []
            for line in f:
                line = line.strip()
                if not line:
                    if buf:
                        trees.append(" ".join(buf))
                        buf = []
                else:
                    buf.append(line)
            if buf:
                trees.append(" ".join(buf))
        return trees

    @staticmethod
    def load_datafile(filename):
        data = []
        with open(filename, 'r') as f:
            lines = f.readlines()
            for i in range(0, len(lines), 3):
                text_left, _, text_right = [s.lower().strip() for s in lines[i].partition("$T$")]
                aspect = lines[i + 1].lower().strip()
                text_right = text_right.replace("$T$", aspect)
                polarity = lines[i + 2].strip()
                data.append([text_left, text_right, aspect, polarity])

        return data

    @staticmethod
    def load_deptype_map(opt):
        deptype_set = set()
        for filename in [opt.train_file, opt.test_file, opt.val_file]:
            filename = "{}.dep".format(filename)
            if os.path.exists(filename) is False:
                continue
            data = ABSADataset.load_depfile(filename)
            for dep_info in data:
                for item in dep_info:
                    deptype_set.add(item['dep'])
        deptype_map = {"none": 0}
        for deptype in sorted(deptype_set, key=lambda x:x):
            deptype_map[deptype] = len(deptype_map)
        return deptype_map

    @staticmethod
    def get_polarity2id():
        polarity_label = ["-1","0","1"]
        return dict([(label, idx) for idx,label in enumerate(polarity_label)])

# 2. Model building blocks

Shared layer classes used by the five GCN modules in `AsaTgcnSem`. None of
these are module-specific — each GCN module instantiates its own
`nn.ModuleList` of one of these layers in `AsaTgcnSem.__init__`.

| Layer class (cell) | Typed? | Used by |
|---|:---:|---|
| `GraphConvolution` (9) | no | SemGCN, LexGCN |
| `TypeGraphConvolution` (10) | yes (edge-type embedding) | SynGCN, KnoGCN, ConstGCN |
| `SemGraphConvolution` (11) | no | (unused in `AsaTgcnSem`; legacy) |
| `SelfAttention` (12) | — | (unused in `AsaTgcnSem`; legacy) |
| `MultiHeadAttention` (13) | — | SemGCN's runtime attention adjacency |

`TypeGraphConvolution.forward` implements the typed aggregation in thesis
Eqs. (4.1)-(4.4): per-edge embedding `e_ij^r` is added to the source-token
representation via `self.dense(dep_embed)` before the attention-weighted
matmul.

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pytorch_transformers import BertPreTrainedModel,BertModel
# XCatGCN layer + cross-example graphs helper
from cross_example import XCatLayer, CrossExampleGraphs
# Self-conditioned gated fusion
from fusion import GatedFusion

In [9]:
class GraphConvolution(nn.Module):
    """
    Simple GCN layer
    """
    def __init__(self, in_features, out_features, bias=True):
        super(GraphConvolution, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        if bias:
            self.bias = nn.Parameter(torch.FloatTensor(out_features))
        else:
            self.register_parameter('bias', None)

    def forward(self, text, adj):
        hidden = torch.matmul(text, self.weight)
        denom = torch.sum(adj, dim=2, keepdim=True) + 1
        output = torch.matmul(adj, hidden) / denom
        if self.bias is not None:
            return output + self.bias
        else:
            return output

In [10]:
class TypeGraphConvolution(nn.Module):
    """
    TGCN Layer
    """
    def __init__(self, in_features, out_features, embedding_dim, bias=True):
        super(TypeGraphConvolution, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.dense = nn.Linear(embedding_dim, in_features, bias=False) 
        if bias:
            self.bias = nn.Parameter(torch.FloatTensor(out_features))
        else:
            self.register_parameter('bias', None)

    def forward(self, text, adj, dep_embed):
        batch_size, max_len, feat_dim = text.shape 
        val_us = text.unsqueeze(dim=2) 
        val_us = val_us.repeat(1, 1, max_len, 1) 
        val_sum = val_us + self.dense(dep_embed) 
        adj_us = adj.unsqueeze(dim=-1) 
        adj_us = adj_us.repeat(1, 1, 1, feat_dim) 
        hidden = torch.matmul(val_sum, self.weight) 
        output = hidden.transpose(1,2) * adj_us 

        output = torch.sum(output, dim=2) 

        if self.bias is not None:
            return output + self.bias
        else:
            return output

In [11]:
class SemGraphConvolution(nn.Module):
    """
    Semantic GCN layer with attention adjacency matrix 
    """
    def __init__(self, in_features, out_features, attention_heads = 1, bias=True):
        super(SemGraphConvolution, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        if bias:
            self.bias = nn.Parameter(torch.FloatTensor(out_features))
        else:
            self.register_parameter('bias', None)

    def forward(self, text, adj):
        hidden = torch.matmul(text, self.weight)
        denom = torch.sum(adj, dim=2, keepdim=True) + 1
        output = torch.matmul(adj, hidden) / denom
        if self.bias is not None:
            return output + self.bias
        else:
            return output

In [12]:
class SelfAttention(nn.Module):
    """
    it could be the functino fro the sintectic module?????
    """
    def __init__(self, input_dim):
        super(SelfAttention, self).__init__()
        self.input_dim = input_dim
        self.query = nn.Linear(input_dim, input_dim)
        self.key = nn.Linear(input_dim, input_dim)
        self.value = nn.Linear(input_dim, input_dim)
        self.softmax = nn.Softmax(dim=2)
        
    def forward(self, x):
        queries = self.query(x)
        keys = self.key(x)
        values = self.value(x) 
        scores = torch.bmm(queries, keys.transpose(1, 2)) / (self.input_dim ** 0.5) 
        attention = self.softmax(scores)
        return attention

In [13]:
class MultiHeadAttention(nn.Module):

    def __init__(self, h, d_model, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        assert d_model % h == 0 # --- devides and return the value of ther reminder --
        #we should have a size of d_odel and h equal each other

        self.d_k = d_model // h # devide with integral result -- rounds the devision
        self.h = h
        self.linears = self.clones(nn.Linear(d_model, d_model), 2)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, mask=None):
        if mask is not None:
            mask = mask[:, :, :query.size(1)]
            mask = mask.unsqueeze(1)
            
        nbatches = query.size(0)
        query, key = [l(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
                             for l, x in zip(self.linears, (query, key))]
        
        attn = self.attention(query, key, mask=mask, dropout=self.dropout)

        return attn
    

    def attention(self, query, key, mask=None, dropout=None):
        d_k = query.size(-1)
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k) 
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        p_attn = F.softmax(scores, dim=-1)
        if dropout is not None:
            p_attn = dropout(p_attn)

        return p_attn
    
    def clones(self, module, N):
        return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

## 3. `AsaTgcnSem` — the HAABSA7GCN model

The main model. It combines up to seven GCN modules and produces a final
classifier through one of three fusion options, selected by `opt.fusion_type`:

- `concat` (the baseline's default): concatenate every active module's
  aspect-pooled output; `fc_single` is `nn.Linear(G*hidden, num_labels)`.
- `gate` (the baseline's legacy 2-module GMU): only valid when exactly two of
  `{tgcn, semgcn, lexgcn, knogcn}` are active.
- `gated` (HAABSA7GCN): module-count-agnostic, self-conditioned
  softmax-across-modules fusion (`GatedFusion` in `fusion.py`); `fc_single`
  collapses to `nn.Linear(hidden, num_labels)`. The per-batch gates are saved
  on `self.last_gates` for the qualitative analysis.

| Module | Adjacency | Layer class | Edge embedding |
|---|---|---|---|
| **SynGCN** (TGCN) | dependency parse (`ABSADataset.create_feature`) | `TypeGraphConvolution` | `dep_embedding` (shared with KnoGCN) |
| **SemGCN** | self-attention over the sequence (per forward) | `GraphConvolution` | — |
| **LexGCN** | co-occurrence lookups (`get_lex_adj`) | `GraphConvolution` | — |
| **KnoGCN** | ontology-filtered dependency parse | `TypeGraphConvolution` | `dep_embedding` (shared with SynGCN) |
| **ConstGCN** | constituency tree, LCA-labelled edges | `TypeGraphConvolution` | `const_embedding` |
| **XCatGCN** | per-instance graph of same-category mentions (`xcat_neighbours`) | `XCatLayer` (`cross_example.py`) | `xcat_category_embedding` (per category) |
| **XSimGCN** | per-instance graph of top-K cosine-similar mentions (`xsim_neighbours`) | `XCatLayer` | `xsim_edge_embedding` (single learned vector) |

Modules are switched on or off through the `opt.modules` dict
(`{'tgcn', 'semgcn', 'lexgcn', 'knogcn', 'constgcn', 'xcatgcn', 'xsimgcn'}`,
each a bool). The first four are the inherited 4GCN baseline; the last three
are the HAABSA7GCN additions.


In [ ]:
class AsaTgcnSem(BertPreTrainedModel):
    def __init__(self, config, modules, tokenizer, opt):
#     use_ensemble = True, fusion_type = 'concat', dropout = 0.2, concat_dropout = 0.5,
#                  cooc_path = 'cooc_matrix_ids.csv', cooc = None):
        """
        modules: dictionary of form {'tgcn': bool, 'semgcn': bool, 'lexgcn': bool, 'knogcn': bool}
        cooc: cooc matrix as dataframe preloaded into memory. if not passed as argument,
        the matrix will be loaded from the specified path.
        """
        
        super(AsaTgcnSem, self).__init__(config)
        self.opt = opt
        self.modules = opt.modules
        
        
        self.use_tgcn, self.use_semgcn, self.use_lexgcn, self.use_knogcn = opt.modules['tgcn'], opt.modules['semgcn'], opt.modules['lexgcn'], opt.modules['knogcn']
        self.use_constgcn = opt.modules.get('constgcn', False)
        self.use_xcatgcn = opt.modules.get('xcatgcn', False)
        self.use_xsimgcn = opt.modules.get('xsimgcn', False)
        self.num_modules = sum((self.use_tgcn, self.use_semgcn, self.use_lexgcn, self.use_knogcn, self.use_constgcn, self.use_xcatgcn, self.use_xsimgcn))
        self.use_ensemble = opt.use_ensemble
        self.layer_number_tgcn = opt.num_layers['tgcn'] 
        self.layer_number_sem = opt.num_layers['semgcn'] 
        self.layer_number_lex = opt.num_layers['lexgcn'] 
        self.layer_number_kno = opt.num_layers['knogcn'] 
        self.layer_number_const = opt.num_layers.get('constgcn', 2)
        self.layer_number_xcat = opt.num_layers.get('xcatgcn', 2)
        self.layer_number_xsim = opt.num_layers.get('xsimgcn', 2)
        

        
        assert self.use_tgcn or self.use_semgcn or self.use_lexgcn or self.use_knogcn or self.use_constgcn or self.use_xcatgcn
        assert opt.fusion_type in ('concat', 'gate', 'gated'), f"unknown fusion_type {opt.fusion_type!r}"  # 'gated' is the new softmax-across-modules fusion
        self.fusion_type = opt.fusion_type
        
        self.num_labels = config.num_labels 
        self.num_types = config.num_types
        
        
        self.bert = BertModel(config)
        

        # SynGCN (TGCN) -- dependency-syntactic GCN
        #   layer:     TypeGraphConvolution
        #   adjacency: dep_adj_matrix from ABSADataset (word->subword projection)
        #   note:      typo blocks gradients (kept for faithful reproduction)
        if self.use_tgcn:
            self.TGCNLayers = nn.ModuleList(([TypeGraphConvolution(config.hidden_size, config.hidden_size, config.hidden_size)
                                             for _ in range(self.layer_number_tgcn)]))
        # SemGCN -- semantic GCN over BERT self-attention
        #   layer:     GraphConvolution (untyped)
        #   adjacency: MultiHeadAttention(seq_out_semgcn, seq_out_semgcn) at runtime
        #   note:      MHA is reinstantiated per forward -> its params never train
        if self.use_semgcn:
            self.SemGCNLayers = nn.ModuleList(([GraphConvolution(config.hidden_size, config.hidden_size)
                                            for _ in range(self.layer_number_sem)]))

            
        # LexGCN -- lexical co-occurrence GCN
        #   layer:     GraphConvolution (untyped)
        #   adjacency: get_lex_adj() from cooc_matrix_final2.csv (subword-native)
        if self.use_lexgcn:
            self.LexGCNLayers = nn.ModuleList(([GraphConvolution(config.hidden_size, config.hidden_size)
                                           for _ in range(self.layer_number_lex)]))

        # KnoGCN -- knowledge GCN over ontology-filtered dep edges
        #   layer:     TypeGraphConvolution
        #   adjacency: dep_adj_matrix_knogcn from ABSADataset (ontology-masked)
        if self.use_knogcn:
            self.KnoGCNLayers = nn.ModuleList(([TypeGraphConvolution(config.hidden_size, config.hidden_size, config.hidden_size)
                                           for _ in range(self.layer_number_kno)]))

        # ConstGCN -- constituency-tree GCN over LCA-labelled edges
        #   layer:     TypeGraphConvolution
        #   adjacency: const_adj_matrix from ABSADataset (.const file, word->subword)
        #   note:      implements layer stacking correctly (not SynGCN's typo)
        if self.use_constgcn:
            self.ConstGCNLayers = nn.ModuleList(([TypeGraphConvolution(config.hidden_size, config.hidden_size, config.hidden_size)
                                             for _ in range(self.layer_number_const)]))

        # XCatGCN -- cross-example category GCN over per-mention star graphs
        #   layer:     XCatLayer (memory-efficient TypeGraphConvolution
        #              specialised to a single-edge-type star)
        #   adjacency: built per-instance in forward from cross_graphs
        #   note:      self.cross_graphs is a plain Python object, not
        #              an nn.Module - call .to(device) on it from the
        #              Instructor since self.to(device) will not move it.
        if self.use_xcatgcn:
            if getattr(opt, "cross_graphs", None) is None:
                raise ValueError("xcatgcn=True requires opt.cross_graphs to be set (a CrossExampleGraphs instance)")
            self.cross_graphs = opt.cross_graphs
            self.XCatGCNLayers = nn.ModuleList([
                XCatLayer(config.hidden_size, config.hidden_size, config.hidden_size)
                for _ in range(self.layer_number_xcat)
            ])
            self.xcat_category_embedding = nn.Embedding(
                self.cross_graphs.num_categories, config.hidden_size
            )

        # XSimGCN -- cross-example similarity GCN over per-mention star
        # graphs whose neighbours are the top-K cosine-similar same-pool mentions in
        # frozen pretrained-BERT space. Same topology as XCat, so XCatLayer is reused.
        # Edge embedding r is a single learned vector (no per-edge typing); we use
        # an nn.Embedding(1, hidden) indexed at 0 so the surface mirrors XCat's
        # category_embedding(cat_id) call pattern.
        if self.use_xsimgcn:
            if getattr(opt, "cross_graphs", None) is None:
                raise ValueError("xsimgcn=True requires opt.cross_graphs to be set (a CrossExampleGraphs instance)")
            # cross_graphs may already have been bound by xcatgcn=True above; same instance.
            self.cross_graphs = opt.cross_graphs
            self.XSimGCNLayers = nn.ModuleList([
                XCatLayer(config.hidden_size, config.hidden_size, config.hidden_size)
                for _ in range(self.layer_number_xsim)
            ])
            # Single learned edge vector for all XSim edges. nn.Embedding(1, d) is
            # used (not nn.Parameter) so .to(device) and state_dict handling are
            # identical to xcat_category_embedding.
            self.xsim_edge_embedding = nn.Embedding(1, config.hidden_size)

       
        
        if self.use_lexgcn:
            
            if opt.cooc is not None: 
                self.cooc = opt.cooc
                
            else:
                self.cooc = pd.read_csv(opt.cooc_path, index_col=0) 
                self.cooc.index = self.cooc.index.astype(int)
                self.cooc.columns = self.cooc.columns.astype(int)
        
        if self.use_knogcn:
            if opt.onto_words is not None:
                self.onto_words = opt.onto_words
            else:
                self.onto_words = pd.read_csv(opt.onto_words_path) 

        # multiplied by num_modules if concat; single hidden_size for the legacy
        # 2-module gate AND the new module-count-agnostic gated fusion.
        if self.fusion_type == 'concat':
            self.fc_single = nn.Linear(config.hidden_size*self.num_modules, self.num_labels)
        elif self.fusion_type == 'gate':
            self.fc_single = nn.Linear(config.hidden_size, self.num_labels)
        elif self.fusion_type == 'gated':
            self.fc_single = nn.Linear(config.hidden_size, self.num_labels)
            self.gated_fusion = GatedFusion(
                num_modules=self.num_modules,
                hidden_dim=config.hidden_size,
                dropout=getattr(opt, 'fusion_dropout', 0.0),
            )
            self.last_gates = None  # populated each forward for qualitative analysis
        
        
        self.gate_weight = nn.Parameter(torch.FloatTensor(config.hidden_size, config.hidden_size * 2))
        self.gate_bias = nn.Parameter(torch.FloatTensor(config.hidden_size))
    
        self.dropout = nn.Dropout(opt.dropout)
        self.concat_dropout = nn.Dropout(opt.concat_dropout)
        self.ensemble_linear_tgcn = nn.Linear(1, self.layer_number_tgcn)
        self.ensemble_linear_semgcn = nn.Linear(1, self.layer_number_sem)
        self.ensemble_linear_lexgcn = nn.Linear(1, self.layer_number_lex)
        self.ensemble_linear_knogcn = nn.Linear(1, self.layer_number_kno)
        self.ensemble_linear_constgcn = nn.Linear(1, self.layer_number_const)
        self.ensemble_linear_xcatgcn = nn.Linear(1, self.layer_number_xcat)
        self.ensemble_linear_xsimgcn = nn.Linear(1, self.layer_number_xsim)

        
        self.ensemble = nn.Parameter(torch.FloatTensor(3, 1))
        self.dep_embedding = nn.Embedding(self.num_types, config.hidden_size, padding_idx=0)
        # ConstGCN edge-type embedding. Vocabulary is CONST_LABEL_VOCAB
        # from cell 6 ({none, self, NP, VP, ADJP, ADVP, PP, SBAR}); dim
        # matches SynGCN's dep_embedding (= config.hidden_size): as in
        # SynGCN, the edge-type embedding is a learned vector
        # attached to each constituency edge.
        self.const_embedding = nn.Embedding(len(constlabel2id), config.hidden_size, padding_idx=0)

    def get_attention(self, val_out, dep_embed, adj):
        batch_size, max_len, feat_dim = val_out.shape
        val_us = val_out.unsqueeze(dim=2)
        val_us = val_us.repeat(1,1,max_len,1)
        val_cat = torch.cat((val_us, dep_embed), -1).float()
        atten_expand = (val_cat * val_cat.transpose(1,2))

        attention_score = torch.sum(atten_expand, dim=-1)
        attention_score = attention_score / np.power(feat_dim, 0.5)
        exp_attention_score = torch.exp(attention_score)
        exp_attention_score = torch.mul(exp_attention_score, adj.float()) # mask
        sum_attention_score = torch.sum(exp_attention_score, dim=-1).unsqueeze(dim=-1).repeat(1,1,max_len)

        attention_score = torch.div(exp_attention_score, sum_attention_score + 1e-10)
        if 'HalfTensor' in val_out.type():
            attention_score = attention_score.half()

        return attention_score
    
    def get_lex_adj(self, input_ids, batch_size, max_len):
        # Initialize an empty adjacency tensor
        adj_tensor = torch.zeros((batch_size, max_len, max_len))
        
        
        # number of non-zero input_ids for each sentence
        num_words = []
        
        # i refers to the sentence number 
        for i, id_sequence in enumerate(input_ids):
            num_words.append(int(torch.sum(id_sequence != 0)))
            
            
            for j in range(num_words[i]):
                for k in range(num_words[i]):
                    if j != k:
                        id_j, id_k = id_sequence[j].item(), id_sequence[k].item()
                        
                        if id_j in self.cooc and id_k in self.cooc:
                            adj_tensor[i, j, k] = self.cooc[id_j][id_k]
                        else:
                            adj_tensor[i, j, k] = 0
            
            
        # Calculate the sums of rows for each matrix
        row_sums = adj_tensor.sum(dim=2, keepdim=True).repeat(1, 1, max_len)

        # Calculate the sums of columns for each matrix
        column_sums = adj_tensor.sum(dim=1, keepdim=True).repeat(1, max_len, 1)

        # Create a diagonal mask for each matrix
        diagonal_mask = torch.eye(adj_tensor.size(-1)).bool().unsqueeze(0).repeat(batch_size, 1, 1)

        total_sum = row_sums + column_sums

        # Set the diagonal entries to the sum of all the row and column entries (will be averaged later)
        res = torch.where(diagonal_mask, total_sum, adj_tensor)
        
        adj_tensor = adj_tensor + res
        
        # Average 
        
        for i, num in enumerate(num_words):
            # Divide diagonal elements by 2
            diagonal = torch.diagonal(adj_tensor[i])
            diagonal_divided = diagonal / num

            # Assign divided diagonal elements back to the tensor
            adj_tensor[i].diagonal().copy_(diagonal_divided)

        return adj_tensor

    def get_avarage(self, aspect_indices, x):
        aspect_indices_us = torch.unsqueeze(aspect_indices, 2)
        x_mask = x * aspect_indices_us
        aspect_len = (aspect_indices_us != 0).sum(dim=1)
        x_sum = x_mask.sum(dim=1)
        x_av = torch.div(x_sum, aspect_len)

        return x_av
    
    def set_dropout(self, dropout):
        self.dropout = nn.Dropout(dropout)

        
    def forward(self, input_ids, segment_ids, valid_ids, mem_valid_ids, dep_adj_matrix, dep_value_matrix, dep_adj_matrix_knogcn,dep_value_matrix_knogcn, const_adj_matrix=None, const_value_matrix=None, xcat_self_feature=None, xcat_category_id=None, xcat_mention_id=None, xcat_split_indicator=None):  # xcat_* kwargs
        # Generate sentence representation with BERT
        sequence_output, pooled_output = self.bert(input_ids, segment_ids)
        
        # Dependency type embeddings
        dep_embed = self.dep_embedding(dep_value_matrix)
        dep_embed_knogcn = self.dep_embedding(dep_value_matrix_knogcn)
        # Constituency edge-type embedding for ConstGCN.
        const_embed = self.const_embedding(const_value_matrix) if self.use_constgcn else None
        
        
        # Initializing valid output tensor (i.e. 0 for padding, only keeping representations of tokens in sentence)
        batch_size, max_len, feat_dim = sequence_output.shape
        valid_output = torch.zeros(batch_size, max_len, feat_dim, device=input_ids.device).type_as(sequence_output)
        
        for i in range(batch_size):
            temp = sequence_output[i][valid_ids[i] == 1]
            valid_output[i][:temp.size(0)] = temp
        valid_output = self.dropout(valid_output)

        attention_score_for_output = [] # Useless code?
        attention_score_knogcn_for_output=[]
        tgcn_layer_outputs = []
        semgcn_layer_outputs = []
        lexgcn_layer_outputs = []
        knogcn_layer_outputs = []
        constgcn_layer_outputs = []
        
        
        seq_out_tgcn = valid_output
        seq_out_semgcn = valid_output
        seq_out_lexgcn = valid_output
        seq_out_knogcn = valid_output
        seq_out_const = valid_output
        
        
        # forward: SynGCN (TGCN) -- dependency-syntactic per-layer aggregation
        if self.use_tgcn:
            for tgcn in self.TGCNLayers:
                # Computing attention
                attention_score = self.get_attention(seq_out_tgcn, dep_embed, dep_adj_matrix)
                
                attention_score_for_output.append(attention_score) # Useless code?

                # Applying GCN layer
                # assignment target now
                # matches the variable read at the next iteration, so layer
                # stacking actually composes. The
                # previous `seq_out = ...` discarded the layer output and
                # blocked gradient flow into TGCNLayers' params.
                seq_out_tgcn = F.relu(tgcn(seq_out_tgcn, attention_score, dep_embed))

                # Saving layer output to be used for layer ensemble later
                tgcn_layer_outputs.append(seq_out_tgcn)
                
            # Average aspect terms for each layer and combining into list 
            tgcn_layer_outputs_pool = [self.get_avarage(mem_valid_ids, x_out) for x_out in tgcn_layer_outputs]

        # forward: SemGCN -- BERT-self-attention per-layer aggregation
        if self.use_semgcn:
            for semgcn in self.SemGCNLayers:
                # Computing attention
                attn = MultiHeadAttention(1, feat_dim)
                attn.to('cuda')
                attn_tensor = attn(seq_out_semgcn, seq_out_semgcn)
                attn_tensor = attn_tensor.squeeze(1)

                # Applying GCN layer
                seq_out_semgcn = F.relu(semgcn(seq_out_semgcn, attn_tensor))

                # Saving layer output
                semgcn_layer_outputs.append(seq_out_semgcn)
                
            # Average aspect terms for each layer and combining into list
            semgcn_layer_outputs_pool = [self.get_avarage(mem_valid_ids, x_out) for x_out in semgcn_layer_outputs]

        
        # forward: LexGCN -- lexical-cooccurrence per-layer aggregation
        if self.use_lexgcn:
            for lexgcn in self.LexGCNLayers:
                # Compute adjaceny matrix
                adj_tensor = self.get_lex_adj(input_ids, batch_size, max_len)
                adj_tensor = adj_tensor.to('cuda')
                # Applying GCN layer
                seq_out_lexgcn = F.relu(lexgcn(seq_out_lexgcn, adj_tensor))
                
                # Saving layer output
                lexgcn_layer_outputs.append(seq_out_lexgcn)
            
            # Average aspect terms for each layer and combining into list
            lexgcn_layer_outputs_pool = [self.get_avarage(mem_valid_ids, x_out) for x_out in lexgcn_layer_outputs]
        
        # forward: KnoGCN -- ontology-filtered dep per-layer aggregation
        if self.use_knogcn:
            for knogcn in self.KnoGCNLayers:
                # Computing attention
                attention_score_knogcn = self.get_attention(seq_out_knogcn, dep_embed_knogcn, dep_adj_matrix_knogcn)
                attention_score_knogcn_for_output.append(attention_score_knogcn) # Useless code?

                # Applying GCN layer
                seq_out_knogcn = F.relu(knogcn(seq_out_knogcn, attention_score_knogcn, dep_embed_knogcn))

                # Saving layer output to be used for layer ensemble later
                knogcn_layer_outputs.append(seq_out_knogcn)
                
            # Average aspect terms for each layer and combining into list 
            knogcn_layer_outputs_pool = [self.get_avarage(mem_valid_ids, x_out) for x_out in knogcn_layer_outputs]
            
        # forward: ConstGCN -- constituency-tree per-layer aggregation
        if self.use_constgcn:
            # ConstGCN: typed-edge graph convolution over the constituency
            # adjacency built in ABSADataset.create_feature.
            # Layer stacking is implemented here -
            # `seq_out_const` IS reassigned each iteration. This diverges
            # from SynGCN's loop, which has a missing reassignment that
            # prevents its parameters from receiving gradients.
            # Otherwise the structure
            # mirrors SynGCN.
            for constgcn in self.ConstGCNLayers:
                attention_score_const = self.get_attention(seq_out_const, const_embed, const_adj_matrix)
                seq_out_const = F.relu(constgcn(seq_out_const, attention_score_const, const_embed))
                constgcn_layer_outputs.append(seq_out_const)
            constgcn_layer_outputs_pool = [self.get_avarage(mem_valid_ids, x_out) for x_out in constgcn_layer_outputs]

        # forward: XCatGCN -- per-instance star-graph aggregation
        # One forward pass per batch item: each item has its own star graph
        # (centre = this mention, leaves = its same-category neighbours in the
        # frozen training pool), so we cannot batch the (1+K)x(1+K) adjacency
        # tensor without padding to the largest category in the batch (memory
        # blow-up). XCatLayer is a closed-form star-graph TypeGCN that needs
        # only O(K*d) memory.
        xcatgcn_layer_outputs_pool = None
        if self.use_xcatgcn:
            B = input_ids.size(0)
            xcat_per_layer = [[] for _ in range(self.layer_number_xcat)]
            for b in range(B):
                mid = int(xcat_mention_id[b].item())
                cat = int(xcat_category_id[b].item())
                split = {0: 'train', 1: 'val', 2: 'test'}[int(xcat_split_indicator[b].item())]
                nbr_idx = self.cross_graphs.xcat_neighbours(split, mid)
                self_x = xcat_self_feature[b].unsqueeze(0)
                if nbr_idx.numel() > 0:
                    nbr_x = self.cross_graphs.train_X[nbr_idx]
                    X_b = torch.cat([self_x, nbr_x], dim=0)
                else:
                    X_b = self_x
                cat_t = torch.tensor(cat, device=X_b.device, dtype=torch.long)
                r_emb = self.xcat_category_embedding(cat_t)
                h = X_b
                for layer_idx, layer in enumerate(self.XCatGCNLayers):
                    h = F.relu(layer(h, r_emb))
                    xcat_per_layer[layer_idx].append(h[0])
            xcatgcn_layer_outputs_pool = [
                torch.stack(layer_outs, dim=0) for layer_outs in xcat_per_layer
            ]
            
        # forward: XSimGCN -- per-instance star-graph aggregation
        # Mirrors XCat's loop exactly. The only differences are:
        #   - neighbour source: cross_graphs.xsim_neighbours (cosine top-K)
        #     instead of cross_graphs.xcat_neighbours (same-category).
        #   - edge embedding r: a single learned vector reused for every centre,
        #     fetched via xsim_edge_embedding(0).
        # xcat_* batch fields are reused as the shared mention identity/feature/
        # split-indicator (Q2 design choice).
        xsimgcn_layer_outputs_pool = None
        if self.use_xsimgcn:
            B = input_ids.size(0)
            # Single learned edge vector — same r for every batch row, every edge.
            r_xsim = self.xsim_edge_embedding(
                torch.zeros(1, dtype=torch.long, device=input_ids.device)
            ).squeeze(0)  # (hidden,)
            xsim_per_layer = [[] for _ in range(self.layer_number_xsim)]
            for b in range(B):
                mid = int(xcat_mention_id[b].item())
                split = {0: 'train', 1: 'val', 2: 'test'}[int(xcat_split_indicator[b].item())]
                nbr_idx = self.cross_graphs.xsim_neighbours(split, mid)
                self_x = xcat_self_feature[b].unsqueeze(0)
                if nbr_idx.numel() > 0:
                    # XSim's neighbour indices always point into train_X (val centres
                    # included, since val mentions are train_X rows held out at runtime).
                    nbr_x = self.cross_graphs.train_X[nbr_idx]
                    X_b = torch.cat([self_x, nbr_x], dim=0)
                else:
                    X_b = self_x
                h = X_b
                for layer_idx, layer in enumerate(self.XSimGCNLayers):
                    h = F.relu(layer(h, r_xsim))
                    xsim_per_layer[layer_idx].append(h[0])
            xsimgcn_layer_outputs_pool = [
                torch.stack(layer_outs, dim=0) for layer_outs in xsim_per_layer
            ]
            
        all_outputs = []
        
        if self.use_ensemble:
            if self.use_tgcn:
                # Layer ensemble for tgcn
                tgcn_pool = torch.stack(tgcn_layer_outputs_pool, -1) # stacking layer outputs 
                ensemble_tgcn = torch.matmul(tgcn_pool, F.softmax(self.ensemble_linear_tgcn.weight, dim=0))
                ensemble_tgcn = ensemble_tgcn.squeeze(dim=-1)
                ensemble_tgcn = self.dropout(ensemble_tgcn)
                all_outputs.append(ensemble_tgcn)
            
            if self.use_semgcn:
                # Layer ensemble for semgcn
                semgcn_pool = torch.stack(semgcn_layer_outputs_pool, -1)
                ensemble_semgcn = torch.matmul(semgcn_pool, F.softmax(self.ensemble_linear_semgcn.weight, dim = 0))
                ensemble_semgcn = ensemble_semgcn.squeeze(dim=-1)
                ensemble_semgcn = self.dropout(ensemble_semgcn)
                all_outputs.append(ensemble_semgcn)
            
            if self.use_lexgcn:
            # Layer ensemble for lexgcn
                lexgcn_pool = torch.stack(lexgcn_layer_outputs_pool, -1)
                ensemble_lexgcn = torch.matmul(lexgcn_pool, F.softmax(self.ensemble_linear_lexgcn.weight, dim = 0))
                ensemble_lexgcn = ensemble_lexgcn.squeeze(dim=-1)
                ensemble_lexgcn = self.dropout(ensemble_lexgcn)
                all_outputs.append(ensemble_lexgcn)
            
            if self.use_knogcn:
            # Layer ensemble for knogcn
                knogcn_pool = torch.stack(knogcn_layer_outputs_pool, -1)
                ensemble_knogcn = torch.matmul(knogcn_pool, F.softmax(self.ensemble_linear_knogcn.weight, dim = 0))
                ensemble_knogcn = ensemble_knogcn.squeeze(dim=-1)
                ensemble_knogcn = self.dropout(ensemble_knogcn)
                all_outputs.append(ensemble_knogcn)

            if self.use_constgcn:
            # Layer ensemble for constgcn
                constgcn_pool = torch.stack(constgcn_layer_outputs_pool, -1)
                ensemble_constgcn = torch.matmul(constgcn_pool, F.softmax(self.ensemble_linear_constgcn.weight, dim = 0))
                ensemble_constgcn = ensemble_constgcn.squeeze(dim=-1)
                ensemble_constgcn = self.dropout(ensemble_constgcn)
                all_outputs.append(ensemble_constgcn)

            if self.use_xcatgcn:
            # Layer ensemble for xcatgcn
                xcatgcn_pool = torch.stack(xcatgcn_layer_outputs_pool, -1)
                ensemble_xcatgcn = torch.matmul(xcatgcn_pool, F.softmax(self.ensemble_linear_xcatgcn.weight, dim = 0))
                ensemble_xcatgcn = ensemble_xcatgcn.squeeze(dim=-1)
                ensemble_xcatgcn = self.dropout(ensemble_xcatgcn)
                all_outputs.append(ensemble_xcatgcn)

            if self.use_xsimgcn:
            # Layer ensemble for xsimgcn
                xsimgcn_pool = torch.stack(xsimgcn_layer_outputs_pool, -1)
                ensemble_xsimgcn = torch.matmul(xsimgcn_pool, F.softmax(self.ensemble_linear_xsimgcn.weight, dim = 0))
                ensemble_xsimgcn = ensemble_xsimgcn.squeeze(dim=-1)
                ensemble_xsimgcn = self.dropout(ensemble_xsimgcn)
                all_outputs.append(ensemble_xsimgcn)
            
        
        else:
            # Take only the last layer output
            if self.use_tgcn:
                ensemble_tgcn = tgcn_layer_outputs_pool[-1]
                all_outputs.append(ensemble_tgcn)
            if self.use_semgcn:
                ensemble_semgcn = semgcn_layer_outputs_pool[-1]
                all_outputs.append(ensemble_semgcn)
            if self.use_lexgcn:
                ensemble_lexgcn = lexgcn_layer_outputs_pool[-1]
                all_outputs.append(ensemble_lexgcn)
            if self.use_knogcn:
                ensemble_knogcn = knogcn_layer_outputs_pool[-1]
                all_outputs.append(ensemble_knogcn)
            if self.use_constgcn:
                ensemble_constgcn = constgcn_layer_outputs_pool[-1]
                all_outputs.append(ensemble_constgcn)
            if self.use_xcatgcn:
                ensemble_xcatgcn = xcatgcn_layer_outputs_pool[-1]
                all_outputs.append(ensemble_xcatgcn)
            if self.use_xsimgcn:
                ensemble_xsimgcn = xsimgcn_layer_outputs_pool[-1]
                all_outputs.append(ensemble_xsimgcn)
            
        # forward: gated fusion -- softmax-across-modules combine
        # Three fusion paths share this point in the forward pass:
        #   - 'gated': self.gated_fusion produces a
        #     (B, hidden) tensor + (B, hidden, G) gates tensor (saved on
        #     self.last_gates for qualitative analysis).
        #   - 'concat' (default): cat across modules -> (B, G*hidden).
        #   - 'gate'   (legacy 2-module GMU below): overrides ensemble_out
        #     in the block immediately after this one; only fires when
        #     num_modules == 2.
        if self.fusion_type == 'gated':
            ensemble_out, _gates = self.gated_fusion(all_outputs)
            self.last_gates = _gates.detach()
        else:
            ensemble_out = torch.cat(all_outputs, dim=1)
        
        
        # gating only if 2 modules used
        # added additional combinations of modules used
        if self.fusion_type == 'gate' and self.num_modules == 2: 
            if self.use_tgcn and self.use_semgcn:
                concatenated = torch.cat((ensemble_tgcn, ensemble_semgcn), dim=1) 
                g = torch.matmul(concatenated, self.gate_weight.t()) + self.gate_bias  # Compute W_g[h0 ; h1] + b_g
                g = torch.sigmoid(g)
                ensemble_out = g * ensemble_tgcn + (1 - g) * ensemble_semgcn
            if self.use_tgcn and self.use_lexgcn:
                concatenated = torch.cat((ensemble_tgcn, ensemble_lexgcn), dim=1) 
                g = torch.matmul(concatenated, self.gate_weight.t()) + self.gate_bias  # Compute W_g[h0 ; h1] + b_g
                g = torch.sigmoid(g)
                ensemble_out = g * ensemble_tgcn + (1 - g) * ensemble_lexgcn
            if self.use_tgcn and self.use_knogcn:
                concatenated = torch.cat((ensemble_tgcn, ensemble_knogcn), dim=1) 
                g = torch.matmul(concatenated, self.gate_weight.t()) + self.gate_bias  # Compute W_g[h0 ; h1] + b_g
                g = torch.sigmoid(g)
                ensemble_out = g * ensemble_tgcn + (1 - g) * ensemble_knogcn
            if self.use_tgcn and self.use_knogcn:
                concatenated = torch.cat((ensemble_tgcn, ensemble_knogcn), dim=1) 
                g = torch.matmul(concatenated, self.gate_weight.t()) + self.gate_bias  # Compute W_g[h0 ; h1] + b_g
                g = torch.sigmoid(g)
                ensemble_out = g * ensemble_tgcn + (1 - g) * ensemble_knogcn
            if self.use_lexgcn and self.use_semgcn:
                concatenated = torch.cat((ensemble_semgcn, ensemble_lexgcn), dim=1) 
                g = torch.matmul(concatenated, self.gate_weight.t()) + self.gate_bias  # Compute W_g[h0 ; h1] + b_g
                g = torch.sigmoid(g)
                ensemble_out = g * ensemble_semgcn + (1 - g) * ensemble_lexgcn
            if self.use_knogcn and self.use_semgcn:
                concatenated = torch.cat((ensemble_semgcn, ensemble_knogcn), dim=1) 
                g = torch.matmul(concatenated, self.gate_weight.t()) + self.gate_bias  # Compute W_g[h0 ; h1] + b_g
                g = torch.sigmoid(g)
                ensemble_out = g * ensemble_semgcn + (1 - g) * ensemble_knogcn
            if self.use_knogcn and self.use_lexgcn:
                concatenated = torch.cat((ensemble_lexgcn, ensemble_knogcn), dim=1) 
                g = torch.matmul(concatenated, self.gate_weight.t()) + self.gate_bias  # Compute W_g[h0 ; h1] + b_g
                g = torch.sigmoid(g)
                ensemble_out = g * ensemble_lexgcn + (1 - g) * ensemble_knogcn
          
        # Additional dropout
        if self.fusion_type == 'concat' and ((self.num_modules == 2) or self.num_modules >= 4):  # guard - gated/legacy-gate fusions skip concat_dropout
            ensemble_out = self.concat_dropout(ensemble_out)
            
        output = self.fc_single(ensemble_out)
        
        return output
    

## 3b. `AsaTgcn` — TGCN-only fallback (legacy)

Standalone SynGCN-only model kept for `model_type='tgcn'` runs. Not used
in the multi-module hybrid path; kept here unchanged for ablation
compatibility with Diana's original ablation scripts.

In [15]:
class AsaTgcn(BertPreTrainedModel):
    
    def __init__(self, config, dropout = 0.2):
        super(AsaTgcn, self).__init__(config)
        self.config = config
        self.layer_number_tgcn = 3
        self.num_labels = config.num_labels
        self.num_types = config.num_types

        self.bert = BertModel(config)
        self.TGCNLayers = nn.ModuleList(([TypeGraphConvolution(config.hidden_size, config.hidden_size, config.hidden_size)
                                         for _ in range(self.layer_number_tgcn)]))
        self.fc_single = nn.Linear(config.hidden_size, self.num_labels)
        self.dropout = nn.Dropout(dropout)
        self.ensemble_linear_tgcn = nn.Linear(1, self.layer_number_tgcn)
        self.ensemble = nn.Parameter(torch.FloatTensor(3, 1))
        self.dep_embedding = nn.Embedding(self.num_types, config.hidden_size, padding_idx=0)

    def get_attention(self, val_out, dep_embed, adj):
        batch_size, max_len, feat_dim = val_out.shape
        val_us = val_out.unsqueeze(dim=2)
        val_us = val_us.repeat(1,1,max_len,1)
        val_cat = torch.cat((val_us, dep_embed), -1).float()
        atten_expand = (val_cat * val_cat.transpose(1,2))

        attention_score = torch.sum(atten_expand, dim=-1)
        attention_score = attention_score / np.power(feat_dim, 0.5)
        exp_attention_score = torch.exp(attention_score)
        exp_attention_score = torch.mul(exp_attention_score, adj.float()) # mask
        sum_attention_score = torch.sum(exp_attention_score, dim=-1).unsqueeze(dim=-1).repeat(1,1,max_len)

        attention_score = torch.div(exp_attention_score, sum_attention_score + 1e-10)
        if 'HalfTensor' in val_out.type():
            attention_score = attention_score.half()

        return attention_score

    def get_avarage(self, aspect_indices, x):
        aspect_indices_us = torch.unsqueeze(aspect_indices, 2)
        x_mask = x * aspect_indices_us
        aspect_len = (aspect_indices_us != 0).sum(dim=1)
        x_sum = x_mask.sum(dim=1)
        x_av = torch.div(x_sum, aspect_len)
        return x_av
    
    def set_dropout(self, dropout):
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, segment_ids, valid_ids, mem_valid_ids, dep_adj_matrix, dep_value_matrix):
        # Generate sentence representation with BERT
        sequence_output, pooled_output = self.bert(input_ids, segment_ids)
        
        # Dependency type embeddings
        dep_embed = self.dep_embedding(dep_value_matrix)
        
        # Initializing valid output tensor (i.e. 0 for padding, only keeping representations of tokens in sentence)
        batch_size, max_len, feat_dim = sequence_output.shape
        valid_output = torch.zeros(batch_size, max_len, feat_dim, device=input_ids.device).type_as(sequence_output)
        for i in range(batch_size):
            temp = sequence_output[i][valid_ids[i] == 1]
            valid_output[i][:temp.size(0)] = temp
        valid_output = self.dropout(valid_output)

        attention_score_for_output = [] 
        tgcn_layer_outputs = []
        semgcn_layer_outputs = []
        seq_out_tgcn = valid_output
        seq_out_semgcn = valid_output
        for tgcn in self.TGCNLayers:
            # Computing attention
            attention_score = self.get_attention(seq_out_tgcn, dep_embed, dep_adj_matrix)
            attention_score_for_output.append(attention_score) # Useless code?
            
            # Applying GCN layer
            seq_out = F.relu(tgcn(seq_out_tgcn, attention_score, dep_embed))
            
            # Saving layer output to be used for layer ensemble later
            tgcn_layer_outputs.append(seq_out_tgcn)
        
        # Average aspect terms for each layer and combining into list
        tgcn_layer_outputs_pool = [self.get_avarage(mem_valid_ids, x_out) for x_out in tgcn_layer_outputs]
        
        # Layer ensemble for tgcn
        tgcn_pool = torch.stack(tgcn_layer_outputs_pool, -1) # stacking layer outputs 
        ensemble_tgcn = torch.matmul(tgcn_pool, F.softmax(self.ensemble_linear_tgcn.weight, dim=0))
        ensemble_tgcn = ensemble_tgcn.squeeze(dim=-1)
        ensemble_tgcn = self.dropout(ensemble_tgcn)
        
        output = self.fc_single(ensemble_tgcn)

        return output

# 4. Training loop

`Instructor` builds the model, splits train/val, runs the optimiser, saves
per-epoch checkpoints under `opt.model_path`, and writes per-epoch eval
predictions for the hybrid (ontology + backup) score computed by
`src/dual_model_eval.ipynb`.

All reported runs use seed 7; per (year, variant) the epoch with the highest
validation accuracy is the reported reproduction (no mean ± std).

In [16]:
import logging
import argparse
import math
import os
import sys
from time import strftime, localtime
import random
import numpy as np
import subprocess

from pytorch_transformers import BertModel, BertConfig
# from data_utils import Tokenizer4Bert, ABSADataset
# from asa_tgcn_model import AsaTgcn

# !pip install scikit-learn
from sklearn import metrics
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split


CONFIG_NAME = 'config.json'
WEIGHTS_NAME = 'pytorch_model.bin'

logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.addHandler(logging.StreamHandler(sys.stdout))

In [ ]:
class Instructor:
    def __init__(self, opt):
        self.opt = opt
        logger.info(opt)
        # Build the cross-example graph helper from the frozen
        # pretrained-BERT pool ./data/cross_features_{year}.pt. Loaded
        # once before model + datasets so both can read opt.cross_graphs.
        # Resolved relative to opt.train_file so notebook CWD is irrelevant.
        # Broadened to either cross-example module; both share the
        # same CrossExampleGraphs instance. If XSim is on, also precompute its
        # top-K cosine tables here, BEFORE random_split runs (set_train_subset
        # will then rebuild the filtered XSim table inside the same call).
        if (opt.modules.get("xcatgcn", False) or opt.modules.get("xsimgcn", False)) and getattr(opt, "cross_graphs", None) is None:
            data_dir = os.path.dirname(opt.train_file) or "."
            opt.cross_graphs = CrossExampleGraphs(year=int(opt.year), data_dir=data_dir).to(opt.device)
            logger.info(f"loaded cross_graphs from {data_dir}: {opt.cross_graphs.describe().splitlines()[0]}")
        # If XSim is enabled, precompute its full-pool top-K tables.
        # Idempotent: calling again with the same K is a no-op (well, re-builds — same result).
        if opt.modules.get("xsimgcn", False) and getattr(opt, 'cross_graphs', None) is not None \
                and opt.cross_graphs.xsim_top_k is None:
            opt.cross_graphs.set_xsim_top_k(int(opt.xsim_top_k))
            logger.info(f"[+XSim] precomputed top-K={opt.xsim_top_k} cosine neighbour tables")
        deptype2id = ABSADataset.load_deptype_map(opt)
        polarity2id = ABSADataset.get_polarity2id()
        logger.info(deptype2id)
        logger.info(polarity2id)
        self.deptype2id = deptype2id
        self.polarity2id = polarity2id
        
        self.vocab_path = os.path.join(opt.bert_model, 'vocab.txt')
        self.tokenizer = Tokenizer4Bert(opt.max_seq_len, opt.bert_model)
        config = BertConfig.from_pretrained("bert-large-uncased") 
        config.num_labels=opt.polarities_dim
        config.num_types=len(self.deptype2id)
        logger.info(config)
        print('check the type of the model...')
        if opt.model_type == 'tgcn': 
            self.model = AsaTgcn.from_pretrained(opt.bert_model, config=config, dropout = opt.dropout)
        else:
            self.model = AsaTgcnSem.from_pretrained(opt.bert_model, config=config, modules = opt.modules,
                                                    tokenizer = self.tokenizer, opt=self.opt) 
#                                                 use_ensemble = opt.use_ensemble,
#                                                     fusion_type = opt.fusion_type, dropout = opt.dropout, 
#                                                     concat_dropout = opt.concat_dropout,
#                                                    cooc_path = opt.cooc_path, cooc = opt.cooc)
        self.model.set_dropout(opt.dropout)
        self.model.to(opt.device)
        
        print('downloading the files...')
        self.fulltrainset = ABSADataset(opt.train_file, self.tokenizer, self.opt, deptype2id=deptype2id)
        self.trainset = ABSADataset(opt.train_file, self.tokenizer, self.opt, deptype2id=deptype2id)
        self.testset = ABSADataset(opt.test_file, self.tokenizer, self.opt, deptype2id=deptype2id)
        
        print('check if the val exist...')
        if os.path.exists(opt.val_file):
            self.valset = ABSADataset(opt.val_file, self.tokenizer, self.opt, deptype2id=deptype2id)
        elif opt.valset_ratio > 0:
            valset_len = int(len(self.trainset) * opt.valset_ratio)
            self.trainset, self.valset = random_split(self.trainset, (len(self.trainset)-valset_len, valset_len))
            # Plumb the runtime train/val split into the XCat graph and
            # retag the val instances' precomputed features so the forward pass can
            # dispatch them to the val pool instead of the train pool.
            if (self.opt.modules.get('xcatgcn', False) or self.opt.modules.get('xsimgcn', False)) and getattr(self.opt, 'cross_graphs', None) is not None:
                import torch as _torch_leakfix
                _train_idx = _torch_leakfix.tensor(self.trainset.indices, dtype=_torch_leakfix.long)
                self.opt.cross_graphs.set_train_subset(_train_idx)
                _base = self.valset.dataset  # underlying ABSADataset; train + val share it
                assert max(self.valset.indices) < len(_base.feature), \
                    'val Subset indices out of range of underlying ABSADataset.feature'
                for _i in self.valset.indices:
                    _base.feature[_i]['xcat_split_indicator'] = _torch_leakfix.tensor(1, dtype=_torch_leakfix.long)
                logger.info(
                    f'[leak-fix] XCat train pool filtered to {len(self.trainset.indices)} '
                    f'true-train mentions; {len(self.valset.indices)} val instances tagged'
                )
        else:
            self.valset = self.testset
        
        print("check device opt.device.type == cuda")
        print("opt.device.type == cuda")
        if opt.device.type == 'cuda':
            logger.info('cuda memory allocated: {}'.format(torch.cuda.memory_allocated(device=opt.device.index)))

    def _print_args(self):
        n_trainable_params, n_nontrainable_params = 0, 0
        for p in self.model.parameters():
            n_params = torch.prod(torch.tensor(p.shape))
            if p.requires_grad:
                n_trainable_params += n_params
            else:
                n_nontrainable_params += n_params
        logger.info('n_trainable_params: {0}, n_nontrainable_params: {1}'.format(n_trainable_params, n_nontrainable_params))
        logger.info('> training arguments:')
        for arg in vars(self.opt):
            logger.info('>>> {0}: {1}'.format(arg, getattr(self.opt, arg)))

    def _reset_params(self):
        for child in self.model.children():
            if type(child) != BertModel:  # skip bert params
                for p in child.parameters():
                    if p.requires_grad:
                        if len(p.shape) > 1:
                            torch.nn.init.xavier_uniform_(p)
                        else:
                            stdv = 1. / math.sqrt(p.shape[0])
                            torch.nn.init.uniform_(p, a=-stdv, b=stdv)

    def save_model(self, save_path, model, args):
        print('function save_model starts...')
        # Save a trained model, configuration and tokenizer
        model_to_save = model.module if hasattr(model, 'module') else model  # Only save the model it-self
        # If we save using the predefined names, we can load using `from_pretrained`
        output_model_file = os.path.join(save_path, WEIGHTS_NAME)
        output_config_file = os.path.join(save_path, CONFIG_NAME)
        torch.save(model_to_save.state_dict(), output_model_file)

        config = model_to_save.config
        config.__dict__["deptype2id"] = self.deptype2id
        config.__dict__["polarity2id"] = self.polarity2id
        with open(output_config_file, "w", encoding='utf-8') as writer:
            writer.write(config.to_json_string())
        output_args_file = os.path.join(save_path, 'training_args.bin')
        torch.save(args, output_args_file)
        subprocess.run(['cp', self.vocab_path, os.path.join(save_path, 'vocab.txt')])

    def _train(self, criterion, optimizer, train_data_loader, val_data_loader, test_data_loader):
        print('function _train starts...')
        max_val_acc = -1
        max_val_f1 = -1
        global_step = 0
        path = None

        model_home = self.opt.model_path 
#         model_home += '-' + strftime("%y%m%d-%H%M", localtime())

        results = {"bert_model": self.opt.bert_model, "batch_size": self.opt.batch_size,
                   "learning_rate": self.opt.learning_rate, "seed": self.opt.seed,
                  "num_epoch": self.opt.num_epoch, "l2reg": self.opt.l2reg,
                  "dropout": self.opt.dropout}
        for epoch in range(self.opt.num_epoch):
            logger.info('>' * 100)
            logger.info('epoch: {}'.format(epoch))
            n_correct, n_total, loss_total = 0, 0, 0
            
            self.model.train() 
            
            for i_batch, t_sample_batched in enumerate(train_data_loader):

                global_step += 1
                optimizer.zero_grad()

                n_correct, n_total, loss_total = self.train_step(optimizer, i_batch, t_sample_batched, criterion, n_correct, n_total, loss_total)
                if global_step % self.opt.log_step == 0:
                    train_acc = n_correct / n_total
                    train_loss = loss_total / n_total
                    logger.info('epoch: {}, loss: {:.4f}, train acc: {:.4f}'.format(epoch, train_loss, train_acc))

                gc.collect()
                torch.cuda.empty_cache() # try optimization
                
            
            #OLD LINES
            val_acc, val_f1 = Instructor._evaluate_acc_f1(self.model, val_data_loader, device=self.opt.device)
            logger.info('>epoch: {}, val_acc: {:.4f}, val_f1: {:.4f}'.format(epoch, val_acc, val_f1))
            results["{}_val_acc".format(epoch)] = val_acc
            results["{}_val_f1".format(epoch)] = val_f1
            saving_path = os.path.join(model_home, "epoch_{}".format(epoch))
            
            if not os.path.exists(saving_path):
                os.makedirs(saving_path)
            if val_acc > max_val_acc or (val_acc == max_val_acc and val_f1 > max_val_f1):
                max_val_acc = val_acc
                max_val_f1 = val_f1
                
                if opt.save_models == 'last':
                    best_path = saving_path
                    best_model = self.model
                elif opt.save_models == 'all':
                    self.save_model(saving_path, self.model, self.opt)
                elif opt.save_models == 'none':
                    pass 

#                 self.model.eval() #old part we don't need that because inside _evaluate_acc_f1 we have model.eval()
                saving_path = os.path.join(model_home, "epoch_{}_eval.txt".format(epoch))
                test_acc, test_f1 = self._evaluate_acc_f1(self.model, test_data_loader, device=self.opt.device,
                                                          saving_path=saving_path)
                logger.info('>> epoch: {}, test_acc: {:.4f}, test_f1: {:.4f}'.format(epoch, test_acc, test_f1))

                results["max_val_acc"] = max_val_acc
                results["test_acc"] = test_acc
                results["test_f1"] = test_f1
            
            output_eval_file = os.path.join(model_home, "eval_results.txt")
            
            with open(output_eval_file, "w") as writer:
                for k,v in results.items():
                    writer.write("{}={}\n".format(k,v))
        
        acc_file = os.path.join(model_home, "acc-{:.4f}".format(test_acc))
        
        if opt.save_models == 'last':
            self.save_model(best_path, best_model, self.opt)
        
        with open(acc_file, 'w') as f:
            f.write(f"accuracy: {test_acc}")
        return max_val_acc, test_acc, test_f1
    
    def train_step(self, optimizer, i_batch, t_sample_batched, criterion, n_correct, n_total, loss_total):
        # t_sample_batched["raw_text"],
        # XCat fields only present when use_xcatgcn=True
        xcat_kwargs = {k: t_sample_batched[k].to(self.opt.device)
                       for k in ("xcat_self_feature","xcat_category_id","xcat_mention_id","xcat_split_indicator")
                       if k in t_sample_batched}
        outputs = self.model(t_sample_batched["input_ids"].to(self.opt.device),
                             t_sample_batched["segment_ids"].to(self.opt.device),
                             t_sample_batched["valid_ids"].to(self.opt.device),
                             t_sample_batched["mem_valid_ids"].to(self.opt.device),
                             t_sample_batched["dep_adj_matrix"].to(self.opt.device),
                             t_sample_batched["dep_value_matrix"].to(self.opt.device),
                             t_sample_batched["dep_adj_matrix_knogcn"].to(self.opt.device),
                             t_sample_batched["dep_value_matrix_knogcn"].to(self.opt.device),
                             # constituency tensors
                             t_sample_batched["const_adj_matrix"].to(self.opt.device),
                             t_sample_batched["const_value_matrix"].to(self.opt.device),
                             **xcat_kwargs)
        targets = t_sample_batched['polarity'].to(self.opt.device)

        loss = criterion(outputs, targets)
        loss.backward()

        optimizer.step()

        n_correct += (torch.argmax(outputs, -1) == targets).sum().item()
        n_total += len(outputs)
        loss_total += loss.item() * len(outputs)

        return n_correct, n_total, loss_total

    @staticmethod
    def _evaluate_acc_f1(model, data_loader, device, saving_path=None):
        model.eval()
        
        n_correct, n_total = 0, 0
        t_targets_all, t_outputs_all = None, None
        
        #model.eval() #the old place

        saving_path_f = open(saving_path, 'w') if saving_path is not None else None

        with torch.no_grad():
            for t_batch, t_sample_batched in enumerate(data_loader):
                t_targets = t_sample_batched['polarity'].to(device)
                t_raw_texts = t_sample_batched['raw_text']
                t_aspects = t_sample_batched['aspect']

                # XCat fields only present when use_xcatgcn=True
                xcat_kwargs = {k: t_sample_batched[k].to(device)
                               for k in ("xcat_self_feature","xcat_category_id","xcat_mention_id","xcat_split_indicator")
                               if k in t_sample_batched}
                t_outputs = model(t_sample_batched["input_ids"].to(device),
                                  t_sample_batched["segment_ids"].to(device),
                                  t_sample_batched["valid_ids"].to(device),
                                  t_sample_batched["mem_valid_ids"].to(device),
                                  t_sample_batched["dep_adj_matrix"].to(device),
                                  t_sample_batched["dep_value_matrix"].to(device),
                                  t_sample_batched["dep_adj_matrix_knogcn"].to(device),
                                  t_sample_batched["dep_value_matrix_knogcn"].to(device),
                                  # constituency tensors
                                  t_sample_batched["const_adj_matrix"].to(device),
                                  t_sample_batched["const_value_matrix"].to(device),
                                  **xcat_kwargs)
                
                n_correct += (torch.argmax(t_outputs, -1) == t_targets).sum().item()
                n_total += len(t_outputs)

                if t_targets_all is None:
                    t_targets_all = t_targets
                    t_outputs_all = t_outputs
                else:
                    t_targets_all = torch.cat((t_targets_all, t_targets), dim=0)
                    t_outputs_all = torch.cat((t_outputs_all, t_outputs), dim=0)

                if saving_path_f is not None:
                    for t_target, t_output, t_raw_text, t_aspect in zip(t_targets.detach().cpu().numpy(),
                                                                        torch.argmax(t_outputs, -1).detach().cpu().numpy(),
                                                                        t_raw_texts, t_aspects):
                        saving_path_f.write("{}\t{}\t{}\t{}\n".format(t_target, t_output, t_raw_text, t_aspect))
        acc = n_correct / n_total
        f1 = metrics.f1_score(t_targets_all.cpu(), torch.argmax(t_outputs_all, -1).cpu(), labels=[0, 1, 2], average='macro', zero_division=0)
        return acc, f1

    def train(self):
        # Loss and Optimizer
        criterion = nn.CrossEntropyLoss()
        _params = filter(lambda p: p.requires_grad, self.model.parameters())
        optimizer = torch.optim.Adam(_params, lr=self.opt.learning_rate, weight_decay=self.opt.l2reg)

        train_data_loader = DataLoader(dataset=self.trainset, batch_size=self.opt.batch_size, shuffle=True)
        test_data_loader = DataLoader(dataset=self.testset, batch_size=self.opt.batch_size, shuffle=True) 
        val_data_loader = DataLoader(dataset=self.valset, batch_size=self.opt.batch_size, shuffle=True) 
        full_train_data_loader = DataLoader(dataset = self.fulltrainset, batch_size = self.opt.batch_size, shuffle=True)

        self._reset_params()
        max_val_acc, test_acc, test_f1 = self._train(criterion, optimizer, train_data_loader, val_data_loader, test_data_loader)
        return max_val_acc, test_acc, test_f1
    
   

In [19]:
def test(opt):
    logger.info(opt)
    config = BertConfig.from_json_file(os.path.join(opt.model_path, CONFIG_NAME))
    logger.info(config)

    tokenizer = Tokenizer4Bert(opt.max_seq_len, opt.model_path)
    if opt.model_type == 'tgcn':
        model = AsaTgcn.from_pretrained(opt.model_path)
    elif opt.model_type == 'tgcn+sem':
        model = AsaTgcnSem.from_pretrained(opt.model_path)
    model.set_dropout(opt.dropout)
    model.to(opt.device)

    deptype2id = config.deptype2id
    logger.info(deptype2id)
    testset = ABSADataset(opt.test_file, tokenizer, opt, deptype2id=deptype2id)
    test_data_loader = DataLoader(dataset=testset, batch_size=opt.batch_size, shuffle=False)
    test_acc, test_f1 = Instructor._evaluate_acc_f1(model, test_data_loader, device=opt.device)
    logger.info('>> test_acc: {:.4f}, test_f1: {:.4f}'.format(test_acc, test_f1))

In [ ]:
def get_args(model_type = 'tgcn', # tgcn, tgcn+sem, tri_gcn
             # Select which modules to use for hybrid model
             tgcn = True,
             semgcn = True, 
             lexgcn = True,
             knogcn = True,
             constgcn = False,  # off by default so 4GCN runs unchanged
             xcatgcn = False,   # off by default; turns on cross-example category GCN
             xsimgcn = False,   # off by default; turns on cross-example similarity GCN
             tgcn_layers = 2,
             semgcn_layers = 2,
             lexgcn_layers = 2,
             knogcn_layers = 2,
             constgcn_layers = 2,
             xcatgcn_layers = 2,
             xsimgcn_layers = 2,
             path = None, 
             year='2015',
             val_file='val.txt',
             log = 'log',
             bert_model='bert-large-uncased', # (change underscore to the dash)
             #bert_model='bert_large_uncased',
             cooc_path = 'cooc_matrix_final2.csv', # Path to co-occurrence matrix file
             cooc = None, # Pandas DataFrame co-occurrence matrix. If not specified, it will be loaded from cooc_path
             onto_words=None,
             onto_words_path='test_ontology_keys.csv',
             learning_rate=2e-5,
             dropout=0.2,
             concat_dropout = 0.4,
             bert_dropout=0.2,
             l2reg=0.01,
             num_epoch=50,
             batch_size=6, 
             log_step=100,
             max_seq_len=100,
             polarities_dim=3,
             device='cuda',
             seed=50,
             valset_ratio=0.2, # the percentage fo the validation set
             do_train=True,
             do_eval=True,
             eval_epoch_num=0,
             fusion_type = 'concat', # 'concat' | 'gate' (legacy 2-module GMU) | 'gated' (softmax-across-modules)
             fusion_dropout = 0.0,    # score-dropout inside gated fusion (Stage-4 tunable)
             xsim_top_k = 20,         # TPE-tunable; default 20
             use_ensemble = True, 
            save_models='last',
            print_sentences = False, #changed to check the results
             optim = 'adam'
            ):
    
    assert model_type == 'tgcn' or model_type == 'tgcn+sem' or model_type == 'tri_gcn'
    
    opt = argparse.Namespace()
    opt.model_type = model_type
    opt.modules = {'tgcn': tgcn, 'semgcn': semgcn, 'lexgcn': lexgcn, 'knogcn': knogcn, 'constgcn': constgcn, 'xcatgcn': xcatgcn, 'xsimgcn': xsimgcn}
    opt.num_layers = {'tgcn': tgcn_layers, 'semgcn': semgcn_layers, 'lexgcn': lexgcn_layers, 'knogcn': knogcn_layers, 'constgcn': constgcn_layers, 'xcatgcn': xcatgcn_layers, 'xsimgcn': xsimgcn_layers}
    
    opt.year = year
    
    fusion = "" if model_type == 'tgcn' else "+" + fusion_type
    opt.train_file = f'data/train{year}restaurant.txt'
    opt.test_file = f'data/test{year}restaurant.txt'
    # discriminate 4gcn vs 5gcn (vs 6/7) by counting active modules,
    # used in opt.model_path so dual_model_eval.ipynb can pick it up
    n_active_gcn = sum((tgcn, semgcn, lexgcn, knogcn, constgcn, xcatgcn, xsimgcn))
    opt.model_path = f'test_models/{n_active_gcn}gcn_{year}{model_type}{fusion}_seed{seed}_reg{l2reg}_drop{dropout}_cdrop{concat_dropout}_lr{learning_rate}_tgcn{tgcn}_semgcn{semgcn}_lexgcn{lexgcn}_knogcn{knogcn}_constgcn{constgcn}_xcatgcn{xcatgcn}_xsimgcn{xsimgcn}_xsimk{xsim_top_k}_epochs{num_epoch}_{optim.lower()}'
#     if model_type == 'tgcn':
#         opt.model_path = f'models/rest_{year}/BERT.L_seed{seed}_reg{l2reg}_drop{dropout}_lr{learning_rate}_epochs{num_epoch}' 
#     elif model_type == 'tgcn+sem':
#         opt.model_path = f'models/rest_{year}/{model_type}/{model_type}_seed{seed}_reg{l2reg}_drop{dropout}_lr{learning_rate}_epochs{num_epoch}'
    if do_eval and not do_train:
        opt.model_path += f'/epoch_{eval_epoch_num}'
    if path:
        opt.model_path = path
    opt.val_file = val_file
    opt.log = log
    opt.bert_model = bert_model
    opt.cooc_path = cooc_path
    opt.cooc = cooc
    opt.onto_words=onto_words
    opt.onto_words_path=onto_words_path
    opt.learning_rate = learning_rate
    opt.dropout = dropout
    opt.concat_dropout = concat_dropout
    opt.bert_dropout = bert_dropout
    opt.l2reg = l2reg
    opt.num_epoch = num_epoch
    opt.batch_size = batch_size
    opt.log_step = log_step
    opt.max_seq_len = max_seq_len
    opt.polarities_dim = polarities_dim
    opt.device = device
    opt.seed = seed
    opt.valset_ratio = valset_ratio
    opt.do_train = do_train
    opt.do_eval = do_eval
    opt.eval_epoch_num = eval_epoch_num
    opt.fusion_type = fusion_type
    opt.fusion_dropout = fusion_dropout
    opt.xsim_top_k = xsim_top_k  # TPE-tunable; default 20
    opt.use_ensemble = True
    opt.save_models = save_models
    opt.print_sent = print_sentences
    opt.optim = optim
    return opt

In [21]:
def set_seed(opt):
    if opt.seed is not None:
        random.seed(opt.seed)
        np.random.seed(opt.seed)
        torch.manual_seed(opt.seed)
        torch.cuda.manual_seed(opt.seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [22]:
def main(opt):
    opt = opt
    set_seed(opt)

    opt.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') \
        if opt.device is None else torch.device(opt.device)
    opt.n_gpu = torch.cuda.device_count()

    if not os.path.exists(opt.log):
        os.makedirs(opt.log)

    log_file = '{}/log-{}.log'.format(opt.log, strftime("%y%m%d-%H%M", localtime()))
    logger.addHandler(logging.FileHandler(log_file))
    
    print('strat of the do train...')
    if opt.do_train:
        ins = Instructor(opt)
        max_val_acc, test_acc, test_f1 = ins.train()
    elif opt.do_eval:
        test(opt)
    
    return max_val_acc, test_acc, test_f1

## Run the full HAABSA7GCN model!

To train the full **7GCN** model directly from this notebook, run the cell below. It enables the three new modules (ConstGCN, XCatGCN, XSimGCN) with gated fusion and the reported optimal hyperparameters (the TPE rank-1 configuration; see
`kaggle_runners/tpe_search_7GCN.ipynb` and `kaggle_runners/final_run_7GCN.ipynb`).

Requires `data/cross_features_{year}.pt` and the `cross_example` / `fusion`
modules reachable from the working directory. Set `year` to `'2015'` or `'2016'`


In [ ]:
year = '2015'   # '2015' or '2016'
seed = 7

opt = get_args(
    model_type='tri_gcn',
    tgcn=True, semgcn=True, lexgcn=True, knogcn=True,
    constgcn=True, xcatgcn=True, xsimgcn=True,
    fusion_type='gated', 
    learning_rate=1.0490678654396277e-05,
    dropout=0.20,
    l2reg=0.0014773338109036142,
    fusion_dropout=0.30,
    concat_dropout=0.2285714285714286,
    xsim_top_k=20,
    num_epoch=15, batch_size=4,
    year=year, seed=seed,
    save_models='none', use_ensemble=False,
    cooc=cooc, onto_words=onto_words,
    do_train=True, do_eval=True,
)
main(opt)